# Thesis - Intelligent Reflecting Surface

## Dataset Generation

This section creates a synthetic dataset for Intelligent Reflecting Surface (IRS) learning.

For each user position $\mathbf{p}=[x,y]^T$, the simplified cascaded channel follows the notation used in the thesis:
$$
h_{\mathrm{eff}}(\mathbf{p},\boldsymbol{\theta})=\mathbf{h}_{\mathrm{IU}}^{H}(\mathbf{p})\,\mathrm{diag}(e^{j\boldsymbol{\theta}})\,\mathbf{h}_{\mathrm{BI}}
$$
where $\boldsymbol{\theta}=[\theta_1,\dots,\theta_M]^T$ is the IRS phase vector.
The element-wise phase-alignment rule is:
$$
\theta_m^{\star}(\mathbf{p})=-\angle\!\left(h_{\mathrm{IU},m}^{*}(\mathbf{p})\,h_{\mathrm{BI},m}\right),\quad m=1,\dots,M
$$
With $K$ quantized levels, phases are projected onto:
$$
\theta_m \in \left\{0,\frac{2\pi}{K},\dots,\frac{2\pi(K-1)}{K}\right\}
$$
Thus each training sample is a pair $(\mathbf{p}^{(i)},\boldsymbol{\theta}^{\star(i)})$.

In [22]:
import numpy as np

In [23]:
def dataset(samples=1000, size=8, K=4):
    """
    Generate synthetic UE positions and brute-force IRS phase labels.

    Args:
        samples: Number of dataset samples (UE positions).
        size: IRS side length. Total IRS elements are M = size**2.
        K: Number of quantized phase levels used in the search grid.

    Returns:
        positions: Array with shape (samples, 2) containing UE coordinates (x, y).
        theta_opt: Array with shape (samples, M) containing the best phase profile found
            for each sample according to the simplified received-power metric.
    """
    # IRS element positions on a 2D grid, with fixed height z = 1.5
    xx, yy = np.meshgrid(np.linspace(8.5, 9.5, size), np.linspace(3.5, 4.5, size), indexing="ij")
    irs_pos = np.column_stack((xx.ravel(), yy.ravel(), np.full(xx.size, 1.5)))
    # Random UE positions in the area: x in [1, 8], y in [1, 7]
    positions = np.hstack((np.random.uniform(1, 8, (samples, 1)), np.random.uniform(1, 7, (samples, 1))))
    # IRS total number of reflecting elements
    M = size**2

    phase_levels = np.linspace(0, 2 * np.pi, K, endpoint=False)
    theta_opt = np.zeros((samples, M))
    for i in range(samples):
        p = np.array([positions[i, 0], positions[i, 1], 1.0]) # UE with z = 1

        # Simplified channels: pathloss approximation + random phase
        h_BI = 10**(-3.5 / 10) * np.exp(1j * np.random.uniform(0,2 * np.pi, M)) # BS-IRS, 3.5 dB
        h_IU = np.exp(-np.linalg.norm(irs_pos - p, axis=1) / 10) * np.exp(1j * np.random.uniform(0, 2 * np.pi, M))  # IRS-UE

        g = h_IU.conj() * h_BI # channel product
        theta = -np.angle(g) # optimal continuous phase
        idx = np.argmin(np.abs(theta[:,None] - phase_levels), axis=1) # quantization (K phase levels)

        theta_opt[i] = phase_levels[idx]

    return positions, theta_opt

## Algorithm

Helper functions used across all algorithms.

The train/test split is defined as
$$
\mathcal{D}_{\mathrm{train}}=\{(\mathbf{x}^{(i)},\boldsymbol{\theta}^{\star(i)})\}_{i=1}^{N_{\mathrm{tr}}},\quad
\mathcal{D}_{\mathrm{test}}=\{(\mathbf{x}^{(i)},\boldsymbol{\theta}^{\star(i)})\}_{i=N_{\mathrm{tr}}+1}^{N},\quad N_{\mathrm{tr}}=\lfloor rN\rfloor
$$
with split ratio $r\in(0,1)$.
The error metric follows the thesis notation:
$$
\mathrm{NMSE}=\frac{\sqrt{\frac{1}{N}\sum_{i=1}^{N}\lVert \boldsymbol{\theta}^{\star(i)}-\hat{\boldsymbol{\theta}}^{(i)} \rVert_2^2}}{\frac{1}{M}\sum_{m=1}^{M}\mathrm{Var}(\theta_m^{\star})}
$$
This normalization makes performance comparable across outputs with different scales.

In [24]:
def train_test_split(x, y, train_ratio=0.8):
    split = int(train_ratio * len(x))
    return x[:split], x[split:], y[:split], y[split:] # xtrain, xtest, ytrain, ytest

def nmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2)) / np.var(y_true, axis=0).mean() # rmse / var


### Multivariate Linear Regression

Input and output are:
$$
\mathbf{x}=[x,y]^T\in\mathbb{R}^2, \qquad \hat{\boldsymbol{\theta}}\in\mathbb{R}^{M}
$$
The regression model is
$$
\hat{\boldsymbol{\theta}}=\mathbf{W}\mathbf{x}+\mathbf{b}, \qquad \mathbf{W}\in\mathbb{R}^{M\times 2},\; \mathbf{b}\in\mathbb{R}^{M}
$$
Parameters are estimated by least squares:
$$
(\hat{\mathbf{W}},\hat{\mathbf{b}})=\arg\min_{\mathbf{W},\mathbf{b}}\sum_{i=1}^{N}\left\lVert \boldsymbol{\theta}^{\star(i)}-(\mathbf{W}\mathbf{x}^{(i)}+\mathbf{b})\right\rVert_2^2
$$
Then each predicted phase is quantized to the hardware levels
$$
\theta_m \leftarrow \arg\min_{q\in\left\{0,\frac{2\pi}{K},\dots,\frac{2\pi(K-1)}{K}\right\}}\left|\hat{\theta}_m-q\right|, \quad m=1,\dots,M
$$

In [25]:
def lg(x_train, x_test, y_train, y_test):
    X = np.concatenate([x_train, np.ones((x_train.shape[0], 1))], axis=1) # Add bias term as an extra column of ones
    theta = np.linalg.pinv(X.T @ X) @ X.T @ y_train # Closed-form least-squares solution

    W, b = theta[:-1], theta[-1]
    y_pred = x_test @ W + b

    return y_pred, nmse(y_test, y_pred)